# Llama depth-1 whole-answer presence probe

Option A for Sec 8.3: the **whole-answer** probe on the depth-1 single-multiply bin (donor solves ~99.8%). Run CELL 1, **restart the kernel**, paste your HF token in CELL 2, run CELL 2. One ~24GB GPU; only the 8B donor is loaded.

In [ ]:
# === CELL 1: install, then RESTART THE KERNEL and run CELL 2 ===
!pip install -q "transformers==4.46.3" accelerate numpy huggingface_hub
!pip uninstall -y torchvision torchaudio
# torchvision is removed before transformers is imported (version mismatch crashes it).
# >>> RESTART THE KERNEL after this cell, then run CELL 2. <<<

In [ ]:
# === CELL 2: whole-answer presence probe on the depth-1 single-multiply Llama bin ===
# Self-contained. Loads ONLY the Llama-3.1-8B donor (no recipient needed for a presence probe).
# The depth-1 bin is the set of single multiplies (operands 12-98, answer <= 999) -- only ~1152
# exist, so we enumerate them all, keep the ones the donor solves, and report the WHOLE-ANSWER
# probe at the graft layer (L17). (The paper's 576-item "pool" was a train-excluded half of this
# same set; the presence probe is identical either way.)
import os, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login

if not hasattr(nn.Module, "set_submodule"):
    def _ssm(self, target, module):
        mod = self
        for a in target.split(".")[:-1]: mod = getattr(mod, a)
        setattr(mod, target.split(".")[-1], module)
    nn.Module.set_submodule = _ssm

# ------------------------------------------------------------------ paste your HF token (Llama is gated)
login("hf_PASTE_YOUR_TOKEN_HERE")
# ------------------------------------------------------------------
DEVICE = "cuda"; torch.manual_seed(0)
DONOR = "meta-llama/Llama-3.1-8B"
L9 = 17                      # validated Llama donor graft layer

tokenizer = AutoTokenizer.from_pretrained(DONOR)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    DONOR, torch_dtype=torch.bfloat16, attn_implementation="eager", low_cpu_mem_usage=True
).to(DEVICE).eval()
print("loaded", DONOR, "| donor layers:", model.config.num_hidden_layers)

# ---- prompt + encoding (identical to the follow-up notebooks) ----
FEWSHOT = ("2 + 5 = 7\n6 * 3 = 18\n4 * 7 + 2 = 30\n9 * 8 = 72\n3 * 4 + 5 = 17\n40 * 20 = 800\n")
def _aprompt(expr): return f"{FEWSHOT}{expr} ="
def _aencode(expr, ans):
    p = tokenizer(_aprompt(expr)).input_ids
    f = tokenizer(_aprompt(expr) + " " + str(ans)).input_ids
    if f[:len(p)] != p or len(f) <= len(p): return None, None
    j = len(p)
    while j < len(f) and not any(c.isdigit() for c in tokenizer.decode([f[j]])): j += 1
    if j >= len(f): return None, None
    return torch.tensor(f[:j]), f[j]

# ---- the depth-1 bin: ALL unique single multiplies (operands 12-98, answer <= 999) ----
def gen_all_d1():
    out = []
    for a in range(12, 99):
        for b in range(12, 99):
            ans = a * b
            if ans > 999: continue
            ids, tk = _aencode(f"{a} * {b}", ans)
            if tk is None: continue
            out.append(dict(expr=f"{a} * {b}", ans=ans, ids=ids, tok=tk))
    return out
d1_pool = gen_all_d1()
random.Random(411).shuffle(d1_pool)
assert len(d1_pool) > 100, f"only {len(d1_pool)} problems generated -- check tokenizer/_aencode"
print(f"unique single-multiply problems = {len(d1_pool)}   (expected ~1152)")

def left_pad(ids_list, pad):
    L = max(t.numel() for t in ids_list)
    ids = torch.full((len(ids_list), L), pad, dtype=torch.long)
    m = torch.zeros((len(ids_list), L), dtype=torch.long)
    for i, t in enumerate(ids_list):
        t = t.flatten(); ids[i, L - t.numel():] = t; m[i, L - t.numel():] = 1
    return ids, m

@torch.inference_mode()
def donor_states_top(prob_ids, batch=16):
    acc, top = [], []
    for i in range(0, len(prob_ids), batch):
        ids, m = left_pad(prob_ids[i:i+batch], tokenizer.pad_token_id)
        o = model(ids.to(DEVICE), attention_mask=m.to(DEVICE), output_hidden_states=True)
        acc.append(o.hidden_states[L9 + 1][:, -1, :].float().cpu())
        top += o.logits[:, -1, :].argmax(-1).cpu().tolist()
    return torch.cat(acc), top

X9, top = donor_states_top([p["ids"] for p in d1_pool])
donor_acc = float(np.mean([top[i] == d1_pool[i]["tok"] for i in range(len(d1_pool))]))
keep = [i for i in range(len(d1_pool)) if top[i] == d1_pool[i]["tok"]]     # donor-solved (first token)
d1 = [d1_pool[i] for i in keep]; X9 = X9[keep]
print(f"donor first-token acc = {donor_acc:.3f}   (paper 0.998)  |  donor-solved n = {len(d1)}")

# ---- probes: train = first half, eval = second half (matches the paper's split) ----
h = len(d1) // 2
ans3 = [str(p["ans"]) for p in d1]        # every answer is 3 digits (144-999)
def digit_probe(y, steps=300):
    Pw = torch.zeros(X9.shape[1], 10, requires_grad=True)
    opt = torch.optim.Adam([Pw], lr=1e-2); mu = X9[:h].mean(0)
    for _ in range(steps):
        opt.zero_grad(); F.cross_entropy((X9[:h] - mu) @ Pw, y[:h]).backward(); opt.step()
    return ((X9[h:] - mu) @ Pw.detach()).argmax(1)

y1 = torch.tensor([int(a[0]) for a in ans3])                 # leading digit
lead = float((digit_probe(y1) == y1[h:]).float().mean())     # sanity: expect ~0.27 (paper 0.271)

joint = torch.ones(len(d1) - h, dtype=torch.bool)            # WHOLE answer (all 3 digits jointly)
for k in range(3):
    yk = torch.tensor([int(a[k]) for a in ans3])
    joint &= (digit_probe(yk) == yk[h:])
whole = float(joint.float().mean())

maj1 = float((y1[h:] == torch.mode(y1[h:]).values).float().mean())   # leading-digit majority baseline
print("=" * 60)
print(f"  leading-digit probe  = {lead:.3f}   (majority {maj1:.3f}; paper 0.271)  [sanity check]")
print(f"  WHOLE-ANSWER probe   = {whole:.3f}   <== put THIS number in Sec 8.3")
print("=" * 60)
print("If donor acc ~ 0.998 and the leading-digit probe ~ 0.27, the bin reproduced correctly.")